In [ ]:
from FlagEmbedding import BGEM3FlagModel

model = BGEM3FlagModel('BAAI/bge-m3',  
                       use_fp16=True) # Setting use_fp16 to True speeds up computation with a slight performance degradation

/home/dong/miniconda3/envs/data-analysis/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
'(MaxRetryError("HTTPSConnectionPool(host='huggingface.co', port=443): Max retries exceeded with url: /BAAI/bge-m3/resolve/main/tokenizer_config.json (Caused by SSLError(SSLEOFError(8, '[SSL: UNEXPECTED_EOF_WHILE_READING] EOF occurred in violation of protocol (_ssl.c:1000)')))"), '(Request ID: 4ce019cd-c8e5-447d-a287-7b4139afdc43)')' thrown while requesting HEAD https://huggingface.co/BAAI/bge-m3/resolve/main/tokenizer_config.json
Retrying in 1s [Retry 1/5].
'(MaxRetryError("HTTPSConnectionPool(host='huggingface.co', port=443): Max retries exceeded with url: /api/resolve-cache/models/BAAI/bge-m3/5617a9f61b028005a4858fdac845db406aefb181/tokenizer_config.json (Caused by SSLError(SSLEOFError(8, '[SSL: UNEXPECTED_EOF_WH

In [8]:
s1 = model.encode(['番茄鸡蛋'])['dense_vecs']
s2 = model.encode(['西红柿牛腩'])['dense_vecs']
similarity = s1 @ s2.T
similarity

array([[0.547974]], dtype=float32)

In [11]:
model.encode(['番茄鸡蛋','西红柿'])['dense_vecs']

array([[-0.00102436, -0.00345295, -0.03211833, ..., -0.01747555,
        -0.0167389 ,  0.03185312],
       [-0.06861047,  0.04997861, -0.01148434, ..., -0.0137142 ,
         0.01390528,  0.03730147]], shape=(2, 1024), dtype=float32)

In [ ]:
WORKING_DIR = './lightrag_storage'

bgem3 = BGEM3FlagModel('BAAI/bge-m3', use_fp16=True) #

async def custom_embed(
        texts:list[str]
) -> np.ndarray:
    """ 参考openai_embed实现，以用到lightrag """
    return bgem3.encode(texts)['dense_vecs']

async def custom_llm_complete(
    prompt,
    system_prompt=None,
    history_messages=None,
    enable_cot: bool = False,
    keyword_extraction=False,
    **kwargs,
) -> str:
    if history_messages is None:
        history_messages = []
    result = await openai_complete_if_cache(
        "inclusionai/ling-2.6-1t:free",  
        prompt,
        system_prompt=system_prompt,
        history_messages=history_messages,
        enable_cot=enable_cot,
        keyword_extraction=keyword_extraction,
        base_url="https://openrouter.ai/api/v1",
        api_key= os.getenv('OPENROUTER_API_KEY') or "",
        **kwargs,
    )
    return result

In [27]:
await custom_embed(['番茄鸡蛋','西红柿'])

array([[-0.00102436, -0.00345295, -0.03211833, ..., -0.01747555,
        -0.0167389 ,  0.03185312],
       [-0.06861047,  0.04997861, -0.01148434, ..., -0.0137142 ,
         0.01390528,  0.03730147]], shape=(2, 1024), dtype=float32)

In [ ]:
rag = LightRAG(
    working_dir=WORKING_DIR,
    embedding_func=EmbeddingFunc(
        embedding_dim= 1024,
        func = custom_embed
    ),
    llm_model_func=custom_llm_complete,
)
    # Clear old data files
files_to_delete = [
    "graph_chunk_entity_relation.graphml",
    "kv_store_doc_status.json",
    "kv_store_full_docs.json",
    "kv_store_text_chunks.json",
    "vdb_chunks.json",
    "vdb_entities.json",
    "vdb_relationships.json",
]

for file in files_to_delete:
    file_path = os.path.join(WORKING_DIR, file)
    if os.path.exists(file_path):
        os.remove(file_path)
        print(f"Deleting old file:: {file_path}")

await rag.initialize_storages() 

with open("./奥利奥冰淇淋.md", "r", encoding="utf-8") as f:
    await rag.ainsert(f.read())